# Day 065 — Exercise 5: End-to-End Integration Test

An **integration test** exercises the whole system as a user would. It's the last gate before deploy: if this passes, the app is shippable.

`run_integration_test` takes an app factory, builds the app, and runs 8 checks covering every major behaviour:

| # | Test | Checks |
|---|------|--------|
| 1 | health | 200 + status='ok' |
| 2 | templates | non-empty list |
| 3 | generate | 200 + content_id |
| 4 | history | count=1 after generate |
| 5 | content get | item found by id |
| 6 | validation | empty prompt → 422 |
| 7 | rate limit | at-limit → 429 |
| 8 | metrics | /metrics with requests key |

In [ ]:
import re, secrets, time
from datetime import datetime
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# ── same components as shipped_app (local copies for this exercise) ────────────
DAILY_LIMITS = {"free": 5, "pro": 500}
TEMPLATES    = {"email": "Write a {tone} email about {topic}."}

def check_rate_limit(usage_count, plan):
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit: return False, "limit"
    return True, ""

class ContentStore:
    def __init__(self): self._store = {}
    def add(self, uid, p, c):
        cid = secrets.token_urlsafe(8)
        self._store[cid] = {"content_id": cid, "user_id": uid,
                             "prompt": p, "content": c}
        return cid
    def get(self, cid): return self._store.get(cid)
    def list_user(self, uid): return [v for v in self._store.values() if v["user_id"]==uid]

class MetricsCollector:
    def __init__(self): self._req=0; self._err=0; self._lat=[]
    def record(self, sc, ms): self._req+=1; (self._err.__class__.__add__) ;         self._err += (1 if sc>=400 else 0); self._lat.append(ms)
    def summary(self): avg=sum(self._lat)/len(self._lat) if self._lat else 0.0;         return {"requests":self._req,"errors":self._err,
                "avg_latency_ms":round(avg,1),"error_rate":round(self._err/self._req if self._req else 0,3)}

def build_app_under_test(plan="free", process_fn=None, initial_usage=0):
    app=FastAPI(); store=ContentStore(); collector=MetricsCollector()
    state={"plan":plan,"usage":initial_usage}
    class _G(BaseModel): prompt:str=Field(min_length=1); user_id:str=Field(min_length=1)
    @app.middleware("http")
    async def mw(req, call_next):
        s=time.monotonic(); r=await call_next(req)
        collector.record(r.status_code,(time.monotonic()-s)*1000); return r
    @app.get("/health")
    def h(): return {"status":"ok","version":"1.1.0"}
    @app.get("/plan")
    def p(): lim=DAILY_LIMITS.get(state["plan"],0); return {"plan":state["plan"],"usage_today":state["usage"],"limit":lim}
    @app.get("/metrics")
    def m(): return collector.summary()
    @app.get("/templates")
    def t(): return {"templates":list(TEMPLATES.keys())}
    @app.post("/generate")
    def g(req:_G):
        ok,reason=check_rate_limit(state["usage"],state["plan"])
        if not ok: raise HTTPException(429,reason)
        ans=process_fn(req.prompt) if process_fn else req.prompt.upper()
        state["usage"]+=1; cid=store.add(req.user_id,req.prompt,ans)
        return {"content_id":cid,"content":ans,"user_id":req.user_id}
    @app.get("/history/{user_id}")
    def hst(user_id:str): items=store.list_user(user_id); return {"user_id":user_id,"count":len(items),"items":items}
    @app.get("/content/{content_id}")
    def gc(content_id:str):
        item=store.get(content_id)
        if item is None: raise HTTPException(404)
        return item
    return app


## Task

Implement `run_integration_test(app_factory) -> dict`:

- Call `app_factory()` to get the FastAPI app
- Wrap in `TestClient(app, raise_server_exceptions=False)`
- Run all 8 checks; collect failures without raising
- For the rate-limit check (7), create a fresh at-limit app via `build_app_under_test(initial_usage=5)`
- Return `{passed, failed, total, failures: list[str]}`

## Your Implementation

In [ ]:
def run_integration_test(app_factory) -> dict:
    """End-to-end smoke test of a writing-assistant-style app.

    app_factory: zero-arg callable → FastAPI app
                 (should use process_fn=str.upper internally)

    Tests performed:
        1. GET /health           → 200 with status='ok'
        2. GET /templates        → 200 with non-empty templates list
        3. POST /generate (ok)   → 200 with content_id
        4. GET /history/{user}   → count=1 after one generate
        5. GET /content/{cid}    → 200, item matches generate response
        6. POST /generate (empty)→ 422 (validation)
        7. Rate-limit test       → build new app with initial_usage=5,
                                   POST /generate → 429
        8. GET /metrics          → 200 with 'requests' key

    Returns:
        {passed: int, failed: int, total: int, failures: list[str]}

    Never raises — all failures collected.
    """
    # TODO: create client, run each test, collect failures, return summary
    raise NotImplementedError


In [ ]:
def run_integration_test(app_factory) -> dict:
    failures = []
    total    = 8

    def check(name, cond, detail=""):
        if not cond:
            failures.append(f"{name}: {detail}" if detail else name)

    app = app_factory()
    c   = TestClient(app, raise_server_exceptions=False)

    # 1. health
    try:
        r = c.get("/health")
        check("health_ok", r.status_code == 200 and
              r.json().get("status") == "ok", f"got {r.status_code}")
    except Exception as e:
        failures.append(f"health_ok ERROR: {e}")

    # 2. templates
    try:
        r = c.get("/templates")
        check("templates_exist",
              r.status_code == 200 and len(r.json().get("templates", [])) > 0)
    except Exception as e:
        failures.append(f"templates_exist ERROR: {e}")

    # 3. generate ok
    cid = None
    try:
        r = c.post("/generate", json={"prompt": "hello", "user_id": "u_test"})
        check("generate_200", r.status_code == 200, f"got {r.status_code}")
        cid = r.json().get("content_id") if r.status_code == 200 else None
    except Exception as e:
        failures.append(f"generate_200 ERROR: {e}")

    # 4. history
    try:
        r = c.get("/history/u_test")
        check("history_count", r.status_code == 200 and
              r.json().get("count", 0) == 1)
    except Exception as e:
        failures.append(f"history_count ERROR: {e}")

    # 5. content get
    try:
        if cid:
            r = c.get(f"/content/{cid}")
            check("content_get", r.status_code == 200)
        else:
            failures.append("content_get SKIP: no cid from generate")
    except Exception as e:
        failures.append(f"content_get ERROR: {e}")

    # 6. validation
    try:
        r = c.post("/generate", json={"prompt": "", "user_id": "u"})
        check("validation_422", r.status_code == 422, f"got {r.status_code}")
    except Exception as e:
        failures.append(f"validation_422 ERROR: {e}")

    # 7. rate limit (separate at-limit app)
    try:
        at_limit = TestClient(
            build_app_under_test(plan="free", process_fn=str.upper,
                                 initial_usage=5),
            raise_server_exceptions=False,
        )
        r = at_limit.post("/generate", json={"prompt": "x", "user_id": "u"})
        check("rate_limit_429", r.status_code == 429, f"got {r.status_code}")
    except Exception as e:
        failures.append(f"rate_limit_429 ERROR: {e}")

    # 8. metrics
    try:
        r = c.get("/metrics")
        check("metrics_ok",
              r.status_code == 200 and "requests" in r.json())
    except Exception as e:
        failures.append(f"metrics_ok ERROR: {e}")

    passed = total - len(failures)
    return {"passed": passed, "failed": len(failures),
            "total": total, "failures": failures}


## Automated checks

In [ ]:
score, total = 0, 4
try:
    def _factory():
        return build_app_under_test(plan="free", process_fn=str.upper)

    result = run_integration_test(_factory)

    # has required keys
    assert all(k in result for k in ("passed","failed","total","failures"))
    score += 1; print("\u2705 result has passed/failed/total/failures keys")

    # total == 8
    assert result["total"] == 8, f"expected total=8, got {result['total']}"
    score += 1; print("\u2705 total == 8 tests")

    # all 8 tests pass for a correct app
    assert result["passed"] == 8, (
        f"Expected 8 passed, got {result['passed']}. "
        f"Failures: {result['failures']}"
    )
    score += 1; print("\u2705 all 8 tests pass for a correct app")

    # failures is a list
    assert isinstance(result["failures"], list)
    score += 1; print("\u2705 failures is a list")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def run_integration_test(app_factory) -> dict:
    failures = []
    total    = 8

    def check(name, cond, detail=""):
        if not cond:
            failures.append(f"{name}: {detail}" if detail else name)

    app = app_factory()
    c   = TestClient(app, raise_server_exceptions=False)

    # 1. health
    try:
        r = c.get("/health")
        check("health_ok", r.status_code == 200 and
              r.json().get("status") == "ok", f"got {r.status_code}")
    except Exception as e:
        failures.append(f"health_ok ERROR: {e}")

    # 2. templates
    try:
        r = c.get("/templates")
        check("templates_exist",
              r.status_code == 200 and len(r.json().get("templates", [])) > 0)
    except Exception as e:
        failures.append(f"templates_exist ERROR: {e}")

    # 3. generate ok
    cid = None
    try:
        r = c.post("/generate", json={"prompt": "hello", "user_id": "u_test"})
        check("generate_200", r.status_code == 200, f"got {r.status_code}")
        cid = r.json().get("content_id") if r.status_code == 200 else None
    except Exception as e:
        failures.append(f"generate_200 ERROR: {e}")

    # 4. history
    try:
        r = c.get("/history/u_test")
        check("history_count", r.status_code == 200 and
              r.json().get("count", 0) == 1)
    except Exception as e:
        failures.append(f"history_count ERROR: {e}")

    # 5. content get
    try:
        if cid:
            r = c.get(f"/content/{cid}")
            check("content_get", r.status_code == 200)
        else:
            failures.append("content_get SKIP: no cid from generate")
    except Exception as e:
        failures.append(f"content_get ERROR: {e}")

    # 6. validation
    try:
        r = c.post("/generate", json={"prompt": "", "user_id": "u"})
        check("validation_422", r.status_code == 422, f"got {r.status_code}")
    except Exception as e:
        failures.append(f"validation_422 ERROR: {e}")

    # 7. rate limit (separate at-limit app)
    try:
        at_limit = TestClient(
            build_app_under_test(plan="free", process_fn=str.upper,
                                 initial_usage=5),
            raise_server_exceptions=False,
        )
        r = at_limit.post("/generate", json={"prompt": "x", "user_id": "u"})
        check("rate_limit_429", r.status_code == 429, f"got {r.status_code}")
    except Exception as e:
        failures.append(f"rate_limit_429 ERROR: {e}")

    # 8. metrics
    try:
        r = c.get("/metrics")
        check("metrics_ok",
              r.status_code == 200 and "requests" in r.json())
    except Exception as e:
        failures.append(f"metrics_ok ERROR: {e}")

    passed = total - len(failures)
    return {"passed": passed, "failed": len(failures),
            "total": total, "failures": failures}
```

**The `check(name, cond)` helper** is the same pattern as `assert_response` from Day 062 — single-line test with a named failure. Wrapping each check in `try/except` means an exception in test 3 doesn't prevent tests 4-8 from running. `failures: list[str]` is more informative than `failed: int` — you see exactly which tests failed, not just how many.

</details>